# LangGraph 실행 흐름 추적

- 구분: 정답 예시
- 원본: [`02_tracing_langgraph.py`](02_tracing_langgraph.py)
- 연결 교안: [4-2-langfuse-observability-and-evaluation.pdf](../../../../docs/raw/Chapter02/day02/4-2-langfuse-observability-and-evaluation.pdf)

원본 Python 코드는 수정 없이 아래의 한 코드 셀에 담았습니다. 설명과 실행 준비는 별도 셀입니다. **모두 실행** 대신 필요한 셀을 직접 선택하세요.


## 학습 목표

1. StateGraph의 노드와 분기를 읽고 노드별 Trace를 확인한다.
2. 코드를 실행하기 전 입력, 출력, 외부 호출과 파일 변경 가능성을 설명한다.
3. CodeMate가 만든 결과를 원본과 교안에 비춰 검토한다.


## 코드를 읽는 순서

고정 날씨 도구 → State → 조건부 엣지 → 그래프 컴파일 → 콜백 순서로 읽는다.

교안은 개념과 목표를 설명합니다. 코드에서 실제 구현된 부분과 TODO를 구분하세요.


## 실행 준비

저장소 안에서 VS Code로 열고 프로젝트 `.venv` 커널을 선택합니다. 의존성은 `uv sync`로 준비합니다. 사내망에서는 사내 LLM 환경을, 사외에서는 허용된 provider를 준비합니다. 자격증명 값은 셀에 적지 않습니다. 아래 준비 셀은 저장소 루트를 찾고 `__file__`을 원본 Python 파일 경로로 설정합니다.


In [ ]:
from pathlib import Path
import os
import sys
_start = Path.cwd().resolve()
_repo = next((p for p in (_start, *_start.parents) if (p / "pyproject.toml").is_file() and (p / "config" / "llm.py").is_file()), None)
if _repo is None:
    raise FileNotFoundError("저장소 안에서 노트북을 열어 주세요.")
os.chdir(_repo)
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
__file__ = str(_repo / 'src/solution/Chapter02/05-observability/02_tracing_langgraph.py')
print(f"실행 기준 폴더: {_repo}")


## 원본 코드 — 직접 실행

**조건과 부작용:** 날씨 값은 고정 예시다. LLM API와 Langfuse 전송이 발생한다. TODO의 콜백 연결은 실행 결과로 확인해야 한다.

준비 셀을 먼저 실행하고 필요한 서비스·설정을 확인하세요. 다음 셀은 `manual-run`으로 표시했습니다. 원본 전체가 한 번에 실행되므로 API 요청, 파일 변경 또는 대기가 생길 수 있습니다.


In [ ]:
"""
LangGraph 트레이싱: StateGraph 실행 흐름 전체 기록하기

핵심 개념:
  - LangGraph 그래프는 여러 노드·엣지로 구성되는데, 트레이싱하면
    어느 노드에서 얼마나 걸렸는지 단계별로 확인 가능
  - with_config()로 그래프 자체에 CallbackHandler를 붙이면
    invoke() 호출마다 자동으로 추적됨

실행:
  python src/solution/Chapter02/05-observability/02_tracing_langgraph.py
"""

from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langfuse import get_client
from langfuse.langchain import CallbackHandler
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()


@tool
def get_weather(city: str) -> str:
    """도시의 현재 날씨를 반환한다."""
    data = {'서울': '맑음 23°C', '부산': '흐림 19°C', '제주': '비 17°C'}
    return data.get(city, f'{city}의 날씨 정보 없음')


tools = [get_weather]
llm = get_llm().bind_tools(tools)


class State(TypedDict):
    messages: Annotated[list, add_messages]


def agent_node(state: State) -> dict:
    return {'messages': [llm.invoke(state['messages'])]}


builder = StateGraph(State)
builder.add_node('agent', agent_node)
builder.add_node('tools', ToolNode(tools))
builder.add_edge(START, 'agent')
builder.add_conditional_edges('agent', tools_condition)
builder.add_edge('tools', 'agent')

# with_config(): 그래프 인스턴스에 기본 config를 고정
# 이후 모든 invoke() / stream() 호출에 CallbackHandler가 자동 적용됨
langfuse_handler = CallbackHandler()
# TODO: 그래프를 컴파일 하면서, with_config 메서드를 통해 callback으로 `langfuse_handler`를 전달하세요.
app = builder.compile().with_config({'callbacks': [langfuse_handler]})

print('=== LangGraph 트레이싱 ===')
inputs = [
    '서울 날씨 알려줘',
    '부산이랑 제주 날씨 비교해줘',
]
for question in inputs:
    result = app.invoke({'messages': [HumanMessage(content=question)]})
    print(f'Q: {question}')
    print(f'A: {result["messages"][-1].content}')
    print()

langfuse = get_client()
langfuse.flush()
print('Tracing에서 확인')


## CodeMate에게 요청하고 검토하기

> START부터 END까지 가능한 실행 경로와 `with_config` 콜백 연결 TODO를 검토해 줘.

CodeMate가 제안한 수정의 근거를 읽고, 실행 가능한 환경에서 출력·Trace·생성 파일을 확인하세요.


## 확인 질문

1. 도구 호출 때 어떤 노드가 더 나타나는가? 고정 날씨와 실시간 날씨는 어떻게 다른가?
2. 실행 전 어떤 환경과 자료를 확인해야 하는가?
3. 결과가 기대와 다르면 어느 단계의 기록부터 볼 것인가?
